In [1]:
from Bio import SeqIO
import json
import pandas as pd
import os
import pathlib

In [2]:
# extract strain name from path, requires '\' path separator
# takes zero-indexed position of strain name in full path as input
def strain_from_path(path, index = 0):
    return str(path).split("\\")[index]


# extract data from genome genbank input file as formatted by antismash output
def gbk_in_data_extraction(gbk_in):

    # initialize list variable for contig accesion IDs
    contig_accession_ID_list = []
    
    # obtain GCF assembly ID from file name
    gbk_split = gbk_in.stem.split('_')
    gcf_sub_index = gbk_split.index("GCF")
    assembly_ID = '_'.join([gbk_split[gcf_sub_index], gbk_split[gcf_sub_index+1]])

    # Genomsequenz soll bei 0 starten
    genome_length = 0

    # Contigsanzahl soll bei 0 starten
    contigs = 0

    # Öffnet das gbk File
    with open(gbk_in) as gbk_handle:
        # Gibt an wie das gbk Files geöffnet wird
        for index, record in enumerate(SeqIO.parse(gbk_handle, "genbank")):

            # Addiere alle Contig Längen zu einer Gesamtgröße
            genome_length += len(record.seq)
            
            #Zähle alle Contigs
            contigs += 1
            
            # add accession IDs to list
            contig_accession_ID_list.append(record.id)

    # return dictionary of obtained data
    return {"assembly_ID": assembly_ID, "genome_length": genome_length, "contigs": contigs, "contig_accession_ID": contig_accession_ID_list}
    
def gbk_out_data_extraction(gbk_out):
    
    # get cluster ID from gbk out file name
    cluster_ID = gbk_out.stem

    # initialize cluster length var
    cluster_length = 0

    # predicted cluster cores information
    core_cluster_IDs = []
    core_cluster_lengths = []
    core_cluster_types = []
    
    cluster_type = None
    
    with open(gbk_out, 'r') as gbk_handle:
        record = SeqIO.read(gbk_handle, "genbank")
        cluster_length = len(record.seq)

        for feature in record.features:
            # obtain cluster type of region
            if feature.type == "region":
                cluster_type = feature.qualifiers["product"]
                
            if feature.type == "proto_core":
                core_cluster_IDs.append('_'.join([cluster_ID, feature.qualifiers["protocluster_number"][0]]))
                core_cluster_lengths.append(abs(feature.location.end - feature.location.start))
                core_cluster_types.append(feature.qualifiers["product"][0])

    return {"cluster_ID": cluster_ID, "cluster_length": cluster_length, "cluster_type": cluster_type, "core_cluster_IDs": core_cluster_IDs, "core_cluster_lengths": core_cluster_lengths, "core_cluster_types": core_cluster_types}

# region gbk output data extraction and merge in temp df
def data_merge_gbk_out_genome(gbk_out_path_list):
    
    # init empty data frame list
    gbk_out_dataframes = []

    # extract data per region output gbk file in list and add as df to df list
    for gbk_out in gbk_out_path_list:
        gbk_out_dataframes.append(pd.DataFrame(gbk_out_data_extraction(gbk_out)))
    
    # merge dfs and return
    df =  pd.concat(gbk_out_dataframes)
    df.reset_index(inplace=True, drop=True)
    return df

# extract mibig closest characterized compound of region to region comparison from json antimash output
# using list of contig IDs from merged gbk_in and gbk_out dfs
# region to region as protocluster to region score origin is unclear and can exceed a value of 1
# which is logically impossible for a relative comparison score (ranging 0-1)
def json_out_data_extraction_genome(json_out_path, contigs_with_cluster):
     with open(json_out_path) as json_handle:
        json_data = json.load(json_handle)

        # initialize empty lists for json output data extractions per genome
        cluster_ID_list = []
        mibig_closest_ID = []
        mibig_closest_organism = []
        mibig_closest_compound = []
        mibig_closest_score = []
        
        # iterate over all "record" entries in json
        # filter for contigs with BGC as in contigs_with_cluster
        for record in json_data["records"]:
            if record["id"] in contigs_with_cluster:
                mibig_region_dict = record["modules"]["antismash.modules.cluster_compare"]["db_results"]["MIBiG"]["by_region"]
                for cluster_key in mibig_region_dict.keys():
                    
                    # format cluster ID as present in gbk files
                    cluster_ID = f'{record["id"]}.region{int(cluster_key):03d}'
                 
                    # take first value in dict as score as score dict is sorted highest to lowest by default
                    region_to_region_score_keys = list(mibig_region_dict[cluster_key]["RegionToRegion_RiQ"]["scores_by_region"].keys())
                    if len(region_to_region_score_keys) > 0:
                        cluster_ID_list.append(cluster_ID)
                        mibig_ref_ID_key = region_to_region_score_keys[0]
                        mibig_closest_score.append(mibig_region_dict[cluster_key]["RegionToRegion_RiQ"]["scores_by_region"][mibig_ref_ID_key])
                        mibig_closest_ID.append(mibig_region_dict[cluster_key]["RegionToRegion_RiQ"]["reference_regions"][mibig_ref_ID_key]["accession"])
                        mibig_closest_organism.append(mibig_region_dict[cluster_key]["RegionToRegion_RiQ"]["reference_regions"][mibig_ref_ID_key]["organism"])
                        mibig_closest_compound.append(mibig_region_dict[cluster_key]["RegionToRegion_RiQ"]["reference_regions"][mibig_ref_ID_key]["description"])

        return pd.DataFrame({"cluster_ID": cluster_ID_list, "mibig_closest_ID": mibig_closest_ID, "mibig_closest_compound": mibig_closest_compound, "mibig_closest_score": mibig_closest_score, "mibig_closest_organism": mibig_closest_organism})

# obtain all antismash-like directories in provided base directory directory recursively
def antismash_dir_list(base_dir):

    directories = []
    antismash_out_dir_list = []
    
    for item in base_dir.glob('**/'):
        if item.is_dir():
            directories.append(item)

    for d in directories:
        gbk_count = 0
        json_count = 0
        index_html_count = 0
        for f in d.glob('*'):
            if f.is_file():
                if str(f).endswith('.gbk'):
                    gbk_count += 1
                elif str(f).endswith('.json'):
                    json_count += 1
                elif str(f).endswith('index.html'):
                    index_html_count += 1
                else:
                    pass
        # define antismash output directory by having at least one .gbk file and exactly one .json and one index.html file
        if gbk_count > 1 and json_count == 1 and index_html_count == 1:
            antismash_out_dir_list.append(d)
    
    return antismash_out_dir_list

# function to return all required antismash in- and output files (gbk in, json out, region.gbk out)
# takes one antismash output directory path
def antismash_in_out_files(antismash_output_dir):
    
    antismash_gbk_in = [f for f in antismash_output_dir.glob('*.gbk') if not f.name.endswith('*region[0-9][0-9][0-9].gbk')][0]
    antismash_gbk_out = [f for f in antismash_output_dir.glob('*region[0-9][0-9][0-9].gbk')]
    antismash_json_out = [f for f in antismash_output_dir.glob('*.json')][0]

    # return dictionary of antismash analysis file paths
    return {"gbk_in_path": antismash_gbk_in, "gbk_out_path_list": antismash_gbk_out, "json_out_path": antismash_json_out}
    
# data accumulation and merging from data extraction functions per genome
# takes antismash output files path dictionary as provided by antismash_in_out_files 
def data_merge_genome(antismash_files_paths, strain_pos_index):
    # strain name
    strain = strain_from_path(antismash_files_paths["gbk_in_path"], index = strain_pos_index)
    
    # input data file extraction
    gbk_in_data_df = pd.DataFrame(gbk_in_data_extraction(antismash_files_paths["gbk_in_path"]))
    gbk_in_data_df["strain"] = strain
    
    # gbk output data extraction
    gbk_out_data_df = data_merge_gbk_out_genome(antismash_files_paths["gbk_out_path_list"])

    # merge gbk in df and gbk out df on contig accession ID
    gbk_out_data_df["contig_accession_ID"] = gbk_out_data_df["cluster_ID"].str.replace(".region[0-9][0-9][0-9]", "", regex = True)
    df_gbk_merge = pd.merge(gbk_in_data_df, gbk_out_data_df, how = "right", on = "contig_accession_ID")

    # list of contigs containing at least one BGC
    contigs_with_cluster = list(df_gbk_merge["contig_accession_ID"][df_gbk_merge["cluster_ID"].notna()].unique())

    # extract data for mibig characterized compounds if found for predicted BGC
    # uses contig ID list constructed above
    df_json_out_data = json_out_data_extraction_genome(antismash_files_paths["json_out_path"], contigs_with_cluster=contigs_with_cluster)

    # merge gbk in/out dataframe and json out df on cluster_ID
    df_merge_genome = pd.merge(df_gbk_merge, df_json_out_data, how = "left", on = "cluster_ID")
    
    return df_merge_genome

# execute antismash data extraction and table build functions on all antismash dirs in given base dir
# antismash out dirs as obtained from downloading all results from https://antismash.secondarymetabolites.org/
# run in online GUI mode with strictness 'relaxed' and all modules executed
# takes position of index where strain info is saved within folder structure, split on backslash
def extract_data_from_basedir(base_dir, strain_pos_index = 6):
    
    # get list of oll antismash dirs in given base directory
    antismash_out_dirs = antismash_dir_list(base_dir)

    # initialize empty list for all genome data dfs obtained below to concat afterwards
    df_list = []
    
    # extract data and build tmp dataframe per genome for each antismash out dir
    for anti_dir in antismash_out_dirs:
        antismash_files = antismash_in_out_files(anti_dir)
        df_list.append(data_merge_genome(antismash_files, strain_pos_index=strain_pos_index))

    # concatenate genome dfs
    df = pd.concat(df_list)
    df.reset_index(inplace = True, drop = True)
    return df


In [ ]:
root_folder = pathlib.Path("C:/Users/PC/Documents/Lea/Bachelorarbeit")

In [3]:
base_dir = pathlib.Path("C:/Users/bt309051/Desktop/antismash_stat_dev/test_data/")

In [4]:
df = extract_data_from_basedir(base_dir)

In [5]:
df

,assembly_ID,genome_length,contigs,contig_accession_ID,strain,cluster_ID,cluster_length,cluster_type,core_cluster_IDs,core_cluster_lengths,core_cluster_types,mibig_closest_ID,mibig_closest_compound,mibig_closest_score,mibig_closest_organism
0,GCF_018221445.1,5723821,33,NZ_PHKP01000002.1,Bacillus cereus DSM 4282,NZ_PHKP01000002.1.region001,25239,betalactone,NZ_PHKP01000002.1.region001_1,5239,betalactone,BGC0000913,MK-8,0.660910,Enterobacter cloacae
1,GCF_018221445.1,5723821,33,NZ_PHKP01000002.1,Bacillus cereus DSM 4282,NZ_PHKP01000002.1.region002,10321,RiPP-like,NZ_PHKP01000002.1.region002_1,321,RiPP-like,BGC0000883,arginomycin,0.306195,Streptomyces arginensis
2,GCF_018221445.1,5723821,33,NZ_PHKP01000002.1,Bacillus cereus DSM 4282,NZ_PHKP01000002.1.region003,10267,RiPP-like,NZ_PHKP01000002.1.region003_1,267,RiPP-like,NaN,NaN,NaN,NaN
3,GCF_018221445.1,5723821,33,NZ_PHKP01000004.1,Bacillus cereus DSM 4282,NZ_PHKP01000004.1.region001,21854,terpene,NZ_PHKP01000004.1.region001_1,1854,terpene,BGC0000888,bacilysin,0.610541,Bacillus sp. CS93
4,GCF_018221445.1,5723821,33,NZ_PHKP01000008.1,Bacillus cereus DSM 4282,NZ_PHKP01000008.1.region001,43582,NRPS-like,NZ_PHKP01000008.1.region001_1,3582,NRPS-like,BGC0002476,enterobactin,0.570081,Escherichia coli str. K-12 substr. MG1655
5,GCF_018221445.1,5723821,33,NZ_PHKP01000009.1,Bacillus cereus DSM 4282,NZ_PHKP01000009.1.region001,23507,LAP,NZ_PHKP01000009.1.region001_1,3507,LAP,BGC0002584,"azolemycin B, azolemycin A, azolemycin D, azol...",0.607289,Streptomyces sp. FXJ1.264
6,GCF_018221445.1,5723821,33,NZ_PHKP01000010.1,Bacillus cereus DSM 4282,NZ_PHKP01000010.1.region001,18753,NI-siderophore,NZ_PHKP01000010.1.region001_1,3708,NI-siderophore,BGC0000942,petrobactin,0.885651,Bacillus anthracis str. Ames
7,GCF_018221445.1,5723821,33,NZ_PHKP01000013.1,Bacillus cereus DSM 4282,NZ_PHKP01000013.1.region001,51749,NRP-metallophore,NZ_PHKP01000013.1.region001_1,11749,NRP-metallophore,BGC0001185,bacillibactin,0.816614,Bacillus velezensis FZB42
8,GCF_018221445.1,5723821,33,NZ_PHKP01000013.1,Bacillus cereus DSM 4282,NZ_PHKP01000013.1.region001,51749,NRPS,NZ_PHKP01000013.1.region001_2,7158,NRPS,BGC0001185,bacillibactin,0.816614,Bacillus velezensis FZB42
9,GCF_018221445.1,5723821,33,NZ_PHKP01000015.1,Bacillus cereus DSM 4282,NZ_PHKP01000015.1.region001,60333,NRPS,NZ_PHKP01000015.1.region001_1,25918,NRPS,BGC0001133,taxlllaid A,0.754487,Xenorhabdus bovienii SS-2004


In [6]:
df.to_csv("testout.csv")